# WindSight — Forecast Performance Analysis

**Comprehensive analysis of UK wind power forecast accuracy**

This notebook analyzes the performance of BMRS wind generation forecasts against
actual generation data for January 2024. It covers:

1. **Error Metrics** — MAE, median error, P99 error
2. **Error vs Forecast Horizon** — How accuracy degrades with longer horizons
3. **Error by Hour of Day** — Temporal patterns in forecast accuracy
4. **Error Distribution** — Statistical distribution of forecast errors
5. **Wind Reliability** — P10, P20 generation levels for demand planning

In [ ]:
import sys
sys.path.insert(0, '..')

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

# Style
plt.style.use('dark_background')
sns.set_palette('Set2')
plt.rcParams.update({
    'figure.figsize': (14, 6),
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.facecolor': '#1a1d28',
    'figure.facecolor': '#12141c',
    'grid.alpha': 0.15,
})

# Connect to DuckDB
DB_PATH = Path('../data/windsight.duckdb')
assert DB_PATH.exists(), f'Database not found at {DB_PATH}. Run pipeline first.'
conn = duckdb.connect(str(DB_PATH), read_only=True)

print('Connected to DuckDB')
print(f"Actual records: {conn.execute('SELECT COUNT(*) FROM actual_generation').fetchone()[0]}")
print(f"Forecast records: {conn.execute('SELECT COUNT(*) FROM wind_forecast').fetchone()[0]}")

## 1. Data Overview

In [ ]:
# Load actual generation data
actual_df = conn.execute("""
    SELECT start_time, generation_mw
    FROM actual_generation
    ORDER BY start_time
""").fetchdf()

actual_df['start_time'] = pd.to_datetime(actual_df['start_time'], utc=True)

print(f'Actual generation: {len(actual_df)} records')
print(f'Date range: {actual_df["start_time"].min()} — {actual_df["start_time"].max()}')
print(f'\nGeneration statistics (MW):')
actual_df['generation_mw'].describe().round(1)

In [ ]:
# Plot actual generation time series
fig, ax = plt.subplots(figsize=(16, 5))
ax.fill_between(actual_df['start_time'], actual_df['generation_mw'],
                alpha=0.3, color='#3b82f6')
ax.plot(actual_df['start_time'], actual_df['generation_mw'],
        color='#3b82f6', linewidth=0.8, label='Actual Wind Generation')
ax.set_title('UK Wind Generation — January 2024')
ax.set_ylabel('Generation (MW)')
ax.set_xlabel('')
ax.legend(loc='upper right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'))
plt.tight_layout()
plt.show()

## 2. Forecast Error Analysis

We compute forecast errors at a 4-hour horizon (default) and analyze their statistical properties.

In [ ]:
def compute_errors(conn, horizon_hours: float) -> pd.DataFrame:
    """Compute forecast errors at a given horizon."""
    query = f"""
    WITH ranked_forecasts AS (
        SELECT
            f.start_time AS timestamp,
            f.generation_mw AS forecast_mw,
            f.publish_time,
            ROW_NUMBER() OVER (
                PARTITION BY f.start_time
                ORDER BY f.publish_time DESC
            ) AS rn
        FROM wind_forecast f
        WHERE f.publish_time <= f.start_time - INTERVAL '{horizon_hours} hours'
    ),
    best_forecast AS (
        SELECT timestamp, forecast_mw, publish_time
        FROM ranked_forecasts
        WHERE rn = 1
    )
    SELECT
        a.start_time AS timestamp,
        a.generation_mw AS actual_mw,
        bf.forecast_mw,
        (bf.forecast_mw - a.generation_mw) AS error_mw,
        ABS(bf.forecast_mw - a.generation_mw) AS abs_error_mw,
        EXTRACT(HOUR FROM a.start_time) AS hour_of_day
    FROM actual_generation a
    INNER JOIN best_forecast bf ON a.start_time = bf.timestamp
    ORDER BY a.start_time;
    """
    df = conn.execute(query).fetchdf()
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    return df

# Default 4h horizon
errors_df = compute_errors(conn, 4.0)
print(f'Matched forecast-actual pairs: {len(errors_df)}')
errors_df.head()

In [ ]:
# Core Error Metrics
abs_errors = errors_df['abs_error_mw']

metrics = {
    'Mean Absolute Error (MAE)': abs_errors.mean(),
    'Median Absolute Error': abs_errors.median(),
    'P99 Absolute Error': abs_errors.quantile(0.99),
    'Root Mean Squared Error (RMSE)': np.sqrt((abs_errors ** 2).mean()),
    'Max Absolute Error': abs_errors.max(),
    'Sample Count': len(abs_errors),
}

print('\n📊 Forecast Error Metrics (4h Horizon)')
print('=' * 45)
for k, v in metrics.items():
    if isinstance(v, int):
        print(f'  {k:32s}: {v:>8,}')
    else:
        print(f'  {k:32s}: {v:>8.1f} MW')

## 3. Error Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Signed error distribution
axes[0].hist(errors_df['error_mw'], bins=60, alpha=0.7,
             color='#60a5fa', edgecolor='#1a1d28', linewidth=0.5)
axes[0].axvline(0, color='#f87171', linewidth=1.5, linestyle='--', label='Zero error')
axes[0].axvline(errors_df['error_mw'].mean(), color='#fbbf24', linewidth=1.5,
                linestyle=':', label=f'Mean bias: {errors_df["error_mw"].mean():.0f} MW')
axes[0].set_title('Signed Error Distribution (Forecast − Actual)')
axes[0].set_xlabel('Error (MW)')
axes[0].set_ylabel('Count')
axes[0].legend()

# Absolute error CDF
sorted_abs = np.sort(abs_errors)
cdf = np.arange(1, len(sorted_abs) + 1) / len(sorted_abs)
axes[1].plot(sorted_abs, cdf * 100, color='#34d399', linewidth=2)
axes[1].axhline(90, color='#fbbf24', linewidth=1, linestyle=':', alpha=0.6, label='P90')
axes[1].axhline(99, color='#f87171', linewidth=1, linestyle=':', alpha=0.6, label='P99')
axes[1].set_title('Absolute Error — Cumulative Distribution')
axes[1].set_xlabel('Absolute Error (MW)')
axes[1].set_ylabel('Percentile (%)')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Error vs Forecast Horizon

How does forecast accuracy degrade as the horizon increases from 0 to 48 hours?

In [ ]:
# Compute MAE at various horizons
horizons = [0, 1, 2, 4, 6, 8, 12, 16, 20, 24, 30, 36, 42, 48]
horizon_results = []

for h in horizons:
    df = compute_errors(conn, h)
    if len(df) > 0:
        horizon_results.append({
            'horizon': h,
            'mae': df['abs_error_mw'].mean(),
            'median': df['abs_error_mw'].median(),
            'p99': df['abs_error_mw'].quantile(0.99),
            'count': len(df),
        })

horizon_df = pd.DataFrame(horizon_results)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(horizon_df['horizon'], horizon_df['mae'], 'o-',
        color='#3b82f6', linewidth=2, markersize=6, label='MAE')
ax.plot(horizon_df['horizon'], horizon_df['median'], 's--',
        color='#34d399', linewidth=2, markersize=5, label='Median')
ax.fill_between(horizon_df['horizon'], 0, horizon_df['mae'],
                alpha=0.1, color='#3b82f6')
ax.set_title('Forecast Error vs Horizon')
ax.set_xlabel('Forecast Horizon (hours)')
ax.set_ylabel('Error (MW)')
ax.legend()
ax.set_xlim(-1, 50)
plt.tight_layout()
plt.show()

horizon_df

## 5. Error by Hour of Day

Are forecasts more accurate at certain times of day?

In [ ]:
hourly_mae = errors_df.groupby('hour_of_day')['abs_error_mw'].agg(['mean', 'count'])
hourly_mae.columns = ['MAE', 'Count']

fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.bar(hourly_mae.index, hourly_mae['MAE'],
              color='#60a5fa', edgecolor='#1a1d28', linewidth=0.5, alpha=0.8)
ax.set_title('Mean Absolute Error by Hour of Day (4h Horizon)')
ax.set_xlabel('Hour (UTC)')
ax.set_ylabel('MAE (MW)')
ax.set_xticks(range(24))
ax.axhline(abs_errors.mean(), color='#f87171', linewidth=1.5,
           linestyle='--', label=f'Overall MAE: {abs_errors.mean():.0f} MW')
ax.legend()
plt.tight_layout()
plt.show()

hourly_mae.round(1)

## 6. Wind Reliability Analysis

Wind power is weather-dependent and intermittent. For grid planning, we need to estimate
how much wind power can **reliably** meet electricity demand.

- **P10**: 90% of the time, wind generation exceeds this level
- **P20**: 80% of the time, wind generation exceeds this level

These are crucial for capacity planning — they represent the "firm" or "reliable" contribution
of wind to the generation mix.

In [ ]:
gen = actual_df['generation_mw']

quantiles = {
    'P10 (90% exceedance)': gen.quantile(0.10),
    'P20 (80% exceedance)': gen.quantile(0.20),
    'P25 (75% exceedance)': gen.quantile(0.25),
    'P50 (Median)': gen.quantile(0.50),
    'Mean': gen.mean(),
    'P75': gen.quantile(0.75),
    'P90': gen.quantile(0.90),
    'Maximum': gen.max(),
}

# Approximate installed capacity
installed_capacity = 30000  # MW (UK wind ~30 GW)
capacity_factor = gen.mean() / installed_capacity

print('\n⚡ Wind Generation Reliability Analysis (Jan 2024)')
print('=' * 55)
for k, v in quantiles.items():
    print(f'  {k:30s}: {v:>10,.0f} MW')
print(f'\n  Capacity Factor:{capacity_factor:>20.1%}')
print(f'\n  ▸ At P10 ({quantiles["P10 (90% exceedance)"]:.0f} MW):')
print(f'    Wind can reliably supply {quantiles["P10 (90% exceedance)"]:.0f} MW')
print(f'    of demand 90% of the time.')
print(f'\n  ▸ At P20 ({quantiles["P20 (80% exceedance)"]:.0f} MW):')
print(f'    Wind can reliably supply {quantiles["P20 (80% exceedance)"]:.0f} MW')
print(f'    of demand 80% of the time.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Generation duration curve
sorted_gen = np.sort(gen)[::-1]
x_pct = np.linspace(0, 100, len(sorted_gen))

axes[0].fill_between(x_pct, sorted_gen, alpha=0.2, color='#3b82f6')
axes[0].plot(x_pct, sorted_gen, color='#3b82f6', linewidth=2)
axes[0].axhline(quantiles['P10 (90% exceedance)'], color='#f87171',
                linewidth=1.5, linestyle='--',
                label=f'P10: {quantiles["P10 (90% exceedance)"]:.0f} MW')
axes[0].axhline(quantiles['P20 (80% exceedance)'], color='#fbbf24',
                linewidth=1.5, linestyle='--',
                label=f'P20: {quantiles["P20 (80% exceedance)"]:.0f} MW')
axes[0].set_title('Wind Generation Duration Curve')
axes[0].set_xlabel('% of Time Generation Exceeds Level')
axes[0].set_ylabel('Generation (MW)')
axes[0].legend(loc='upper right')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'))

# Generation distribution
axes[1].hist(gen, bins=50, alpha=0.7, color='#60a5fa',
             edgecolor='#1a1d28', linewidth=0.5)
axes[1].axvline(quantiles['P10 (90% exceedance)'], color='#f87171',
                linewidth=1.5, linestyle='--',
                label=f'P10: {quantiles["P10 (90% exceedance)"]:.0f} MW')
axes[1].axvline(quantiles['P20 (80% exceedance)'], color='#fbbf24',
                linewidth=1.5, linestyle='--',
                label=f'P20: {quantiles["P20 (80% exceedance)"]:.0f} MW')
axes[1].axvline(quantiles['Mean'], color='#34d399',
                linewidth=1.5, linestyle=':',
                label=f'Mean: {quantiles["Mean"]:.0f} MW')
axes[1].set_title('Wind Generation Distribution')
axes[1].set_xlabel('Generation (MW)')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Actual vs Forecast Comparison

In [ ]:
# Plot actual vs forecast (first week of Jan)
fig, ax = plt.subplots(figsize=(16, 6))

# Select first week for clarity
mask = errors_df['timestamp'] < '2024-01-08'
week_df = errors_df[mask]

ax.plot(week_df['timestamp'], week_df['actual_mw'],
        color='#3b82f6', linewidth=1.5, label='Actual Generation', alpha=0.9)
ax.plot(week_df['timestamp'], week_df['forecast_mw'],
        color='#34d399', linewidth=1.5, linestyle='--', label='Forecast (4h horizon)', alpha=0.9)
ax.fill_between(week_df['timestamp'],
                week_df['actual_mw'], week_df['forecast_mw'],
                alpha=0.15, color='#f87171', label='Error')
ax.set_title('Actual vs Forecast — First Week of January 2024')
ax.set_ylabel('Generation (MW)')
ax.legend(loc='upper right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'))
plt.tight_layout()
plt.show()

## 8. Summary & Conclusions

### Key Findings

1. **Forecast accuracy** degrades predictably with longer horizons
2. **Error distribution** shows the forecast is approximately unbiased
3. **Reliable supply**: Wind can reliably contribute P10 MW to meet demand 90% of the time
4. **Capacity factor** reflects the intermittent nature of wind power

### Implications for Grid Planning

- **Baseload contribution**: The P10 level represents the "firm" wind contribution.  
  Grid operators should not count on more than this level being available.
- **Forecast improvement**: Shorter horizons significantly improve accuracy.  
  Investing in real-time forecasting yields measurable benefits.
- **Complementary sources**: Given the intermittency shown, wind must be paired  
  with dispatchable generation or storage to ensure grid reliability.

In [ ]:
conn.close()
print('\n✅ Analysis complete. Database connection closed.')